In [1]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("../database/n100.db")

In [2]:
df = pd.read_csv("../reports/financial_ratios_day12.csv")

df.head()

,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio
0,ABB,Dec 2012,22.411128,0.0,NaN,1.822492,17.241379
1,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
2,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
3,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755
4,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755


In [3]:
df.isnull().sum()

company_id                0
year                      0
return_on_equity_pct      0
debt_to_equity            0
interest_coverage        93
asset_turnover            1
dividend_payout_ratio     4
dtype: int64

In [4]:
df = df.fillna(0)

In [5]:
df.isnull().sum()

company_id               0
year                     0
return_on_equity_pct     0
debt_to_equity           0
interest_coverage        0
asset_turnover           0
dividend_payout_ratio    0
dtype: int64

In [6]:
positive_metrics = [
    "return_on_equity_pct",
    "interest_coverage",
    "asset_turnover",
    "dividend_payout_ratio"
]

negative_metrics = [
    "debt_to_equity"
]

for col in positive_metrics:
    df[col + "_score"] = (
        (df[col] - df[col].min()) /
        (df[col].max() - df[col].min())
    ) * 100

for col in negative_metrics:
    df[col + "_score"] = (
        (df[col].max() - df[col]) /
        (df[col].max() - df[col].min())
    ) * 100

df.head()

,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio,return_on_equity_pct_score,interest_coverage_score,asset_turnover_score,dividend_payout_ratio_score,debt_to_equity_score
0,ABB,Dec 2012,22.411128,0.0,0.0,1.822492,17.241379,5.882665,49.483999,0.514884,28.866249,100.0
1,ABB,Mar 2014,25.126904,0.0,0.0,1.998244,12.626263,5.896974,49.483999,0.564537,28.743878,100.0
2,ABB,Mar 2014,25.126904,0.0,0.0,1.998244,12.626263,5.896974,49.483999,0.564537,28.743878,100.0
3,ABB,Mar 2015,24.439701,0.0,0.0,1.665939,12.663755,5.893353,49.483999,0.470655,28.744872,100.0
4,ABB,Mar 2015,24.439701,0.0,0.0,1.665939,12.663755,5.893353,49.483999,0.470655,28.744872,100.0


In [7]:
score_cols = [c for c in df.columns if c.endswith("_score")]

print(score_cols)

['return_on_equity_pct_score', 'interest_coverage_score', 'asset_turnover_score', 'dividend_payout_ratio_score', 'debt_to_equity_score']


In [8]:
df["overall_score"] = df[score_cols].mean(axis=1)

df[[
    "company_id",
    "year",
    "overall_score"
]].head()

,company_id,year,overall_score
0,ABB,Dec 2012,36.949559
1,ABB,Mar 2014,36.937878
2,ABB,Mar 2014,36.937878
3,ABB,Mar 2015,36.918576
4,ABB,Mar 2015,36.918576


In [9]:
df["company_rank"] = (
    df["overall_score"]
    .rank(method="dense", ascending=False)
)

df[[
    "company_id",
    "year",
    "overall_score",
    "company_rank"
]].head()

,company_id,year,overall_score,company_rank
0,ABB,Dec 2012,36.949559,269.0
1,ABB,Mar 2014,36.937878,291.0
2,ABB,Mar 2014,36.937878,291.0
3,ABB,Mar 2015,36.918576,320.0
4,ABB,Mar 2015,36.918576,320.0


In [10]:
top10 = df.sort_values(
    by="overall_score",
    ascending=False
).head(10)

top10[[
    "company_id",
    "year",
    "overall_score",
    "company_rank"
]]

,company_id,year,overall_score,company_rank
1172,INDIGO,Mar 2013,75.534251,1.0
1173,INDIGO,Mar 2014,65.985096,2.0
233,BEL,Mar 2015,63.065817,3.0
231,BEL,Mar 2013,60.577325,4.0
232,BEL,Mar 2014,59.174537,5.0
234,BEL,Mar 2016,56.639958,6.0
1174,INDIGO,Mar 2015,54.922835,7.0
235,BEL,Mar 2017,53.884873,8.0
1019,TATACONSUM,Mar 2016,51.080481,9.0
242,BEL,Mar 2024,49.695588,10.0


In [11]:
ranking_report = df.sort_values(
    by="company_rank"
)

ranking_report.to_csv(
    "../reports/company_ranking_day13.csv",
    index=False
)

print("Company Ranking Report Saved!")

Company Ranking Report Saved!


In [12]:
import os

print(os.listdir("../reports"))

['benchmark_companies.csv', 'benchmark_companies_day7.csv', 'capital_allocation_day11.csv', 'company_ranking_day13.csv', 'financial_ratios_day12.csv', 'highest_price_stocks.csv', 'peer_group_distribution.csv', 'sector_distribution.csv', 'top_market_cap_companies.csv', 'top_market_cap_day7.csv', 'top_roce_companies.csv', 'top_roe_companies.csv', 'top_volume_stocks.csv']
